# This tutorial will show you how to upload "subject sets" from the Roman Research Nexus (RRN) to a Zooniverse project using Panoptes
- [Zooniverse](https://www.zooniverse.org/) is the worlds largest and most widely used platform for hosting people-powered research projects (citizen science)
- The [Panoptes](https://panoptes-python-client.readthedocs.io/en/latest/index.html) package allows you to manage Zooniverse projects through python
- A [Subject](https://help.zooniverse.org/getting-started/#subjects) is a unit of data to be classified or analyzed.
- A [Subject set](https://help.zooniverse.org/getting-started/#subject-sets) is a group of data units (images), along with their metadata ("manifest").  
- More information on getting started with Zooniverse can be found [here](https://help.zooniverse.org/getting-started/)

### In this tutorial, you will learn how to 
- 📡 connect to Zooniverse from within a python environment using Panoptes
- 📁 create a new project linked to your Zooniverse account
- 🌠 create a new subject set and link subjects to the project

This tutorial only covers uploading subject sets from the RRN to Zooniverse, and assumes the user has already created subject set images that are stored on the RRN. Alternatively, if the notebook is run as-is, it will link a set of test subjects hosted externally (on github) to the new Zooniverse project you create. 

On the Zooniverse.org side, you can then login and create a workflow, which is a series of tasks you would like people to complete for each subject in your project (questions or drawing tasks). 

## Getting started
### Create Zooniverse Account
Navigate to [https://www.zooniverse.org/](https://www.zooniverse.org/) and click register in the top right corner to create an account.

### Access the Roman Research Nexus with a MyST account
Anyone with a Space Telescope Science Institute (STScI) [MyST account](https://proper.stsci.edu/proper/authentication/auth) can launch a [Roman Research Nexus](https://roman-docs.stsci.edu/data-handbook/roman-research-nexus) session. 

### Installing panoptes_client
The panoptes_client does not come pre-installed on the RRN. To install the panoptes_client package, you will need to [create a new python environment](https://roman.science.stsci.edu/user/ksand017/lab/tree/references/roman_notebooks/markdown/software.md) on your RRN account. Then you can install panoptes by following [these instructions](https://panoptes-python-client.readthedocs.io/en/latest/user_guide.html#installation). You will also need to install pandas, astropy, tqdm, and fastparquet: ```pip install panoptes_client pandas astropy tqdm fastparquet```

Alternatively, you can setup the new python environment using the provided YAML file. To do this from the terminal, change directory (cd) to the location of the repository contents and run ```kernel-create rrn-2-zooniverse-env rrn-2-zooniverse-specs.yaml "RRN-2-Zooniverse-env"```



In [1]:
from panoptes_client import Panoptes, Workflow, Project, Subject, SubjectSet

import pandas as pd
import numpy as np

from astropy.table import Table as t

import getpass

import os

from tqdm import tqdm
from tqdm import tqdm_notebook
from tqdm import trange


ModuleNotFoundError: No module named 'panoptes_client'

In [ ]:


def update_txt_file(filename, id_field):
    """
    Used for tracking which sources have been 
    uploaded to the Zooniverse project. Appends 
    a new id_field to filename.
    
    Parameters:
    -----------
    filename: str
        the name of the file containing a list 
        of uploaded subjects

    id_field: int, float, str
        The data unit identifier of the 
        uploaded subject
        
    """
    with open(filename, 'a') as f:
        f.write(str(id_field) + '\n')

def get_uploaded_ids(filename):
    """
    Retrieves the list of data unit identifiers 
    that have been uploaded to the Zooniverse 
    project.

    Parameters:
    -----------
    filename: str
        the name of the file containing a list 
        of uploaded subjects
        
    """
    with open(filename, 'r') as f:
        uploaded_ids = [line.strip() for line in f]
    return np.array(uploaded_ids)

def check_if_uploaded(filename, id_field):
    """
    Checks if the given id_field is already in filename.

    Parameters:
    -----------
    filename: str
        the name of the file containing a list 
        of uploaded subjects

    id_field: int, float, str
        The data unit identifier of the 
        uploaded subject
    
    """
    try:
        with open(filename, 'r') as f:
            uploaded_ids = {line.strip() for line in f}
        return str(id_field) in uploaded_ids
    except FileNotFoundError:
        return False  # If the file doesn't exist yet, nothing has been uploaded



### This tutorial:
As it is written, running this notebook will link a set of 20 subjects hosted on a github page to a Zooniverse project. To run the tutorial, you need to download the accompanying tutorial catalog, which contains the metadata for these 20 subjects along with the Amazon Web Services (AWS) location of the calibrated L3 asdf file each subject is detected in. Once you download this catalog, the tutorial proceeds in 3 steps:
- Create manifest file
- Connect to Zooniverse using Panoptes and create a new project and subject set
- Link the 20 subjects to the new subject set



## Creating Our Manifest File
The "manifest" is an important component of uploading a "subject set" and contains a list of the subjects and their properties. The manifest must be formatted as a CSV files and, at minimum, must contain the file path(s) associated with each subject. In addition to file locations, the manifest may also contain metadata linking each subject back to the original data. Our manifest entries will include the locations of the subject images to be uploaded, the RA and DEC locations of each subject, links to external servers that may provide additional information on each subject, and a subject identifier that can be used to link subjects back to the original data.


Some helpful links: 
- [What is a manifest](https://help.zooniverse.org/getting-started/example/#details-subject-sets-and-manifest-details-aka-what-is-a-manifest)
- [FAQ: Metadata, manifests, & more](https://docs.google.com/document/d/16kAtOeFvYYUbxY_1Yi9lhFQ2lvhFNL8-w7khUe7T2fs/edit?tab=t.0)
- [Example manifest file for multi-image subjects](https://docs.google.com/spreadsheets/d/1t8vlOpryNWksWuY-2XToB6EEa_CmnZxh2hHTXzp6GeM/edit?gid=0#gid=0)


In [ ]:
# load in the source catalog that contains your subjects' metadata information 
catalog_name = 'tutorial_catalog_file_locs_f106_f129_f158'
cat_df = pd.read_parquet(catalog_name+'.parquet')

In [ ]:
# define the location where the images are hosted
# this should point to where the individual 
# subject images are saved. To point to subject images on the RRN, 
# replace this url with the path to the RRN-hosted images. 
# e.g., subject_png_save_loc = '/teams/gz-roman/subjects_pngs_test/'
subject_png_save_loc = 'https://ksand017.github.io/GZRoman_HLWAS_test_images/'

# if keeping external github-hosted location, leave this True. 
# If images are hosted locally (on the RRN AWS filesystem), set to false
# this dictates whether the notebook checks the image location to make 
# sure the file exists locally, and affects how individual subjects are 
# linked to the project
external_host = True

In [ ]:
# Create a master catalog of image file locations, source catalog metadata, and any external links you want associate with your subjects

master_cat_metadata = 'tutorial-colour_cutout_master_catalogue_w_metadata.csv'

# don't overwrite the catalog if it already exists
if not os.path.isfile(master_cat_metadata):
    
    image1_paths = []
    image2_paths = []
    ras = []
    decs = []
    desi_links = []
    ned_links = []
    esasky_links = []
    
    for src_i in trange(len(cat_df)):

        # define image names for each subject
        id_str = cat_df['id_str'][src_i]
        image1_paths.append(str(id_str) + '-composite.png')
        image2_paths.append(str(id_str) + '-single_band.png')

        # grab ra,dec information to create external position-based links for each subject 
        ra = np.round(cat_df.loc[id_str]['ra_centroid_f106'],8)
        dec = np.round(cat_df.loc[id_str]['dec_centroid_f106'],8)

        
        desi_link = '[Click to view in DESI](+tab+https://www.legacysurvey.org/viewer?ra='+str(ra)+'&dec='+str(dec)+'&layer=ls-dr9&zoom=15)'
        ned_link = '[Click to search NASA NED](+tab+https://ned.ipac.caltech.edu/conesearch?search_type=Near%20Position%20Search&in_csys=Equatorial&in_equinox=J2000&ra='+str(ra)+'d&dec='+str(dec)+'d&radius=0.1)'
        esasky_link = '[Click to view in ESA Sky](+tab+https://sky.esa.int/esasky/?target='+str(ra)+'%'+str(dec)+'&hips=JWST+NIRCam+Imaging&fov=0.3940858692337921&projection=TAN&cooframe=J2000&sci=false&lang=en)'
    
        ras.append(ra)
        decs.append(dec)
        desi_links.append(desi_link)
        ned_links.append(ned_link)
        esasky_links.append(esasky_link)

    # append this subject-associated metadata to the master catalog and save as a new CSV. We will take manifest entries from this new catalog.
    cat_df['image1_path'] = image1_paths
    cat_df['image2_path'] = image2_paths
    cat_df['!RA'] = ras
    cat_df['!DEC'] = decs
    cat_df['!desi_link'] = desi_links
    cat_df['!ned_link'] = ned_links
    cat_df['!esasky_link'] = esasky_links
    
    cat_df['#id_str'] = cat_df['id_str']
    
    cat_df.to_csv(master_cat_metadata, index=False)

### Manifest columns
Next we need to define which of the columns in our master catalog should be included as manifest items in the subject upload. The format of the column names allows you to specify what information is shared with the volunteers participating in your project. 
- Any fields with names that begin with “#” or “//” will never be shown to volunteers, but will still be returned to you in the classification file, allowing you to link classified subjects back to the original data.
- Any fields with names that begin with "!" will not be accessible to volunteers in the classification interface, but will be available on Talk after classification. 

In [ ]:
# define the columns from this new catalog that should be included in the manifest entries
manifest_columns = ['image1_path',
       'image2_path', '!RA', '!DEC', '!desi_link', '!ned_link', '!esasky_link',
       '#id_str']

## Creating a new project and project-linked subject set using Panoptes
In order to make build a new Zooniverse project, or make changes to an existing project, you must authenticate with Zooniverse using ```Panoptes.connect()```


In [ ]:
Panoptes.connect(username=getpass.getpass('username: '), password=getpass.getpass('password: '))

After logging in, you can create new, or edit existing, projects associated with your Zooniverse account. 

In [ ]:
# to create a new project
tutorial_project = Project()

# edit project information
tutorial_project.display_name = 'RRN Tutorial Project2'
tutorial_project.description = 'My first project created in Python using Panoptes'
tutorial_project.primary_language = 'en'
tutorial_project.private = True

# note that the project will not appear on your Zooniverse.org account unless you save these changes. 
tutorial_project.save()
tutorial_project_id = tutorial_project.id

### Verify the new project was created! 
Log into Zooniverse.org and click "Build" from the top left menu options. You should see the project has been created under Your Projects.

In [ ]:
# to load and edit an existing project, you just need to know the project id
tutorial_project_id = tutorial_project.id # project id indicated on your project page, for now use the id of the project we just created
tutorial_project = Project.find(tutorial_project_id)

We can now create a new subject set to link to the project we just created. 

In [ ]:
# to create a new subject set associated with your project
new_subject_set_name = 'Roman-simulated-HLWAS-sources-tutorial'

print(f'Attempting to edit project: {tutorial_project.title}')
print(f'Creating new subject set titled: {new_subject_set_name}')

tutorial_subject_set = SubjectSet()
tutorial_subject_set.links.project = tutorial_project
tutorial_subject_set.display_name = new_subject_set_name
tutorial_subject_set.save()
tutorial_subject_set_id = tutorial_subject_set.id
print(f'New subject id is: {tutorial_subject_set_id}')

## Link subjects to the subject set
Before we start uploading subjects to our subject set, lets create a file to keep track of which subjects have been uploaded. 

In [ ]:
# For tracking which subjects have already been uploaded
# Create the file initially (if it doesn't exist)
tracking_filename = 'tutorial_uploaded_subjects_by_id_field.txt'

# Make sure the file exists
open(tracking_filename, 'x').close()

In [ ]:
# select which galaxies should be uploaded to the subject set. For now, let's upload all 20 galaxies in our catalog.
selected_galaxies = cat_df.copy()

In [ ]:
# prepare the manifest
manifest = []
uploaded_ids = get_uploaded_ids(tracking_filename) #keeps track of which src ids have already been uploaded via Panoptes
print('uploaded_ids', uploaded_ids)


for i in range(len(selected_galaxies)):
    row = selected_galaxies.loc[i]
    id_str = str(row['#id_str'])

    if id_str in uploaded_ids:
        continue

    entry = {
        'png_locs': [
            subject_png_save_loc + str(row['image1_path']),
            subject_png_save_loc + str(row['image2_path'])
        ],
        'metadata': {col: str(row[col]) for col in manifest_columns},
        'id_str': id_str
    }
    print('entry', entry)
    manifest.append(entry)

print('Manifest created')

In [ ]:
# Upload selected galaxies to Galaxy Zoo 

# Connect once globally
client = Panoptes.client(username=getpass.getpass('username: '), password=getpass.getpass('password: '))

# lookup project and subject set using IDs
project = Project.find(tutorial_project_id)
subject_set = SubjectSet.find(tutorial_subject_set_id)

# for each entry (subject) in manifest...
for scr_i in trange(len(manifest)):
    manifest_item = manifest[src_i]

    # create a new (empty) subject and link the project to the subject
    new_subject = Subject()
    new_subject.links.project = tutorial_project_id

    # for each image to be included for a given subject, use add_location to add the image source
    for img_loc in manifest_item['png_locs']:
        if external_host == False:
            assert os.path.exists(img_loc), f"Image not found: {img_loc}"
            new_subject.add_location(img_loc)
        else:
            new_subject.add_location({'image/png': img_loc})

    # add manifest metadata for the subject
    new_subject.metadata.update(manifest_item['metadata'])

    # save these updates
    new_subject.save(client=client)

    # add the subject to the project-linked subject set
    try:
        subject_set.add(new_subject) 
    except:
        subject_set.reload()
        subject_set.add(new_subject)

    # add the subject to the tracking file
    update_txt_file(tracking_filename, manifest_item['id_str'])


print("All uploads completed.")

## Subject uploads complete!
You can now log onto Zooniverse.org and create a [workflow](https://help.zooniverse.org/getting-started/#workflow-details) for your subject set. 